# Affordable Produce and Recipe Recommendation
## Data Collection, Cleaning, and Exploratory Analysis

### Project Objective
The goal of this project is to investigate produce prices and recipe ingredients to support budget-friendly recipe recommendations.


## Imports and project setup

In [10]:
from pathlib import Path
from getpass import getpass
from requests.auth import HTTPBasicAuth

import os
import re
import json
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "report"

for folder in [RAW_DIR, PROCESSED_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Report directory:", REPORT_DIR)

Project directory: /mnt/c/Users/benny/csci_code/csci_413/produce-recommender
Raw data directory: /mnt/c/Users/benny/csci_code/csci_413/produce-recommender/data/raw
Processed data directory: /mnt/c/Users/benny/csci_code/csci_413/produce-recommender/data/processed
Report directory: /mnt/c/Users/benny/csci_code/csci_413/produce-recommender/report


## Check CSV files in raw data folder can be accessed

In [11]:
raw_files = sorted(RAW_DIR.glob("*.csv"))

print("CSV files found:", len(raw_files))

for file_path in raw_files:
    print(f"\n{'=' * 60}")
    print("Filename:", file_path.name)

    try:
        preview = pd.read_csv(file_path)
        print("Columns:", preview.columns.tolist())
        display(preview.shape)
    except Exception as error:
        print("Could not read this CSV:", error)

CSV files found: 3

Filename: PP_recipes.csv
Columns: ['id', 'i', 'name_tokens', 'ingredient_tokens', 'steps_tokens', 'techniques', 'calorie_level', 'ingredient_ids']


(178265, 8)


Filename: fruit-prices-2023.csv
Columns: ['Fruit', 'Form', 'AverageRetailPrice', 'AverageRetailPriceUnitOfMeasure', 'PreparationYieldFactor', 'SizeOfACupEquivalent ', 'CupEquivalentUnitOfMeasure', 'AveragePricePerCupEquivalent']


(62, 8)


Filename: vegetable-prices-2023.csv
Columns: ['Vegetable', 'Form', 'AverageRetailPrice', 'AverageRetailPriceUnitOfMeasure', 'PreparationYieldFactor', 'SizeOfACupEquivalent ', 'CupEquivalentUnitOfMeasure', 'AveragePricePerCupEquivalent']


(93, 8)

## Use Kroger API to collect store locations

In [13]:
client_id = os.getenv("KROGER_CLIENT_ID")
client_secret = os.getenv("KROGER_CLIENT_SECRET")

if not client_id:
    client_id = getpass("Kroger client ID (input hidden): ")

if not client_secret:
    client_secret = getpass("Kroger client secret (input hidden): ")

zip_code = input("ZIP code for your grocery search: ").strip()

TOKEN_URL = "https://api.kroger.com/v1/connect/oauth2/token"
BASE_URL = "https://api.kroger.com/v1"

token_response = requests.post(
    TOKEN_URL,
    auth=HTTPBasicAuth(client_id, client_secret),
    data={
        "grant_type": "client_credentials",
        "scope": "product.compact"
    },
    timeout=30
)

print("Token status:", token_response.status_code)
token_response.raise_for_status()

access_token = token_response.json()["access_token"]

headers = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/json"
}

location_response = requests.get(
    f"{BASE_URL}/locations",
    headers=headers,
    params={
        "filter.zipCode.near": zip_code,
        "filter.limit": 10
    },
    timeout=30
)

print("Location status:", location_response.status_code)
location_response.raise_for_status()

locations = location_response.json().get("data", [])
location_rows = []

for location in locations:
    address = location.get("address", {})

    location_rows.append({
        "location_id": location.get("locationId"),
        "store_name": location.get("name"),
        "address": address.get("addressLine1"),
        "city": address.get("city"),
        "state": address.get("state"),
        "zip_code": address.get("zipCode")
    })

stores_df = pd.DataFrame(location_rows)
display(stores_df)

if stores_df.empty:
    raise ValueError(
        "No locations returned."
    )

Kroger client ID (input hidden):  ········
Kroger client secret (input hidden):  ········
ZIP code for your grocery search:  80236


Token status: 200
Location status: 200


,location_id,store_name,address,city,state,zip_code
0,62000019,King Soopers - BEAR VALLEY,3100 S Sheridan Blvd,Denver,CO,80227
1,62000109,King Soopers - Federal and Belleview,5050 S Federal Blvd,Englewood,CO,80110
2,62000056,King Soopers - W Evans Ave,2727 W Evans Ave,Denver,CO,80219
3,62000065,King Soopers - Englewood Pkwy and S Broadway,101 Englewood Pkwy,Englewood,CO,80110
4,62000007,King Soopers - W Florida Ave,5125 W Florida Ave,Denver,CO,80219
5,62000031,King Soopers - Jewell and Wadsworth,1927 S Wadsworth Blvd,Lakewood,CO,80227
6,62000131,King Soopers - Littleton and Broadway,100 W Littleton Blvd,Littleton,CO,80120
7,62000050,King Soopers - Governors Plaza,9820 W Belleview Ave,Littleton,CO,80123
8,62000111,King Soopers Fresh Fare - Fresh Fare,3495 S University Blvd,Englewood,CO,80113
9,62000041,King Soopers - Coalmine and Pierce,6760 S Pierce St,Littleton,CO,80128


### Selecting a store and loading datasets

In [14]:
store_index = np.random.default_rng(42).integers(len(stores_df))

if not 0 <= store_index < len(stores_df):
    raise ValueError("Invalid store row number.")

store_id = stores_df.iloc[store_index]["location_id"]

if pd.isna(store_id):
    raise ValueError("The selected store does not have a location ID.")

store_id = str(store_id)

stores_df.to_csv(RAW_DIR / "kroger_stores.csv", index=False)

print("Selected store:", stores_df.iloc[store_index]["store_name"])
print("Store ID:", store_id)

search_terms = [
    "bananas",
    "apples",
    "potatoes",
    "carrots",
    "broccoli",
    "onions",
    "tomatoes",
    "bell pepper"
]

product_rows = []

for term in search_terms:
    response = requests.get(
        f"{BASE_URL}/products",
        headers=headers,
        params={
            "filter.term": term,
            "filter.locationId": store_id,
            "filter.limit": 10
        },
        timeout=30
    )

    print(f"{term}: HTTP {response.status_code}")
    response.raise_for_status()

    products = response.json().get("data", [])

    for product in products:
        for item in product.get("items", []):
            price = item.get("price") or {}

            product_rows.append({
                "search_term": term,
                "product_id": product.get("productId"),
                "upc": product.get("upc"),
                "description": product.get("description"),
                "brand": product.get("brand"),
                "size": item.get("size"),
                "regular_price": price.get("regular"),
                "promo_price": price.get("promo"),
                "store_id": store_id,
                "collection_date": pd.Timestamp.now().date().isoformat()
            })

    time.sleep(0.25)

kroger_raw = pd.DataFrame(product_rows)

if not kroger_raw.empty:
    kroger_raw.to_csv(
        RAW_DIR / "kroger_produce_raw.csv",
        index=False
    )

    print("Products collected:", len(kroger_raw))
    display(kroger_raw.head(10))
else:
    print("No product rows were returned.")

Selected store: King Soopers - BEAR VALLEY
Store ID: 62000019
bananas: HTTP 200
apples: HTTP 200
potatoes: HTTP 200
carrots: HTTP 200
broccoli: HTTP 200
onions: HTTP 200
tomatoes: HTTP 200
bell pepper: HTTP 200
Products collected: 79


,search_term,product_id,upc,description,brand,size,regular_price,promo_price,store_id,collection_date
0,bananas,0000000004011,0000000004011,Fresh Bunch of Bananas – 5-7 Bananas,Fresh Bananas,1 lb,0.55,NaN,62000019,2026-10-14
1,bananas,0000000094011,0000000094011,Fresh Bunch of Organic Bananas – 5-7 Bananas,Fresh Bananas,1 lb,0.79,NaN,62000019,2026-10-14
2,bananas,0000000004235,0000000004235,Fresh Plantain - Single,NaN,1 ea,0.67,NaN,62000019,2026-10-14
3,bananas,0008259219415,0008259219415,Naked Boosted Smoothie Strawberry Banana Machine,Naked,15.2 fl oz,3.79,NaN,62000019,2026-10-14
4,bananas,0008259219464,0008259219464,Naked Boosted Smoothie Strawberry Banana Machine,Naked,64 fl oz,8.99,NaN,62000019,2026-10-14
5,bananas,0001111011379,0001111011379,Simple Truth® Freeze-Dried Bananas,Simple Truth,1.5 oz,4.99,NaN,62000019,2026-10-14
6,bananas,0001111063660,0001111063660,Simple Truth® Berries and Banana Trail Mix,Simple Truth,16 oz,8.99,NaN,62000019,2026-10-14
7,bananas,0001397100502,0001397100502,Bare® Baked Crunchy Simply Banana Chips Dried Fruit Snack,Bare,2.7 oz,6.29,NaN,62000019,2026-10-14
8,bananas,0000000004757,0000000004757,Banana Squash,NaN,1 lb,NaN,NaN,62000019,2026-10-14
9,apples,0000000003283,0000000003283,Large Honeycrisp Apple – Each,Fresh Apples,1 lb,2.49,2.19,62000019,2026-10-14


In [15]:
raw_datasets = {}
load_errors = []

for file_path in sorted(RAW_DIR.glob("*.csv")):
    if file_path.name == "kroger_stores.csv":
        continue

    try:
        df = pd.read_csv(file_path, low_memory=False)
        raw_datasets[file_path.stem] = df

        print(
            f"Loaded {file_path.name}: "
            f"{df.shape[0]:,} rows and {df.shape[1]:,} columns"
        )

    except Exception as error:
        load_errors.append({
            "file": file_path.name,
            "error": str(error)
        })

if load_errors:
    print("\nFiles that could not be loaded:")
    display(pd.DataFrame(load_errors))

if not raw_datasets:
    print(
        "\nNo datasets loaded."
    )

Loaded PP_recipes.csv: 178,265 rows and 8 columns
Loaded fruit-prices-2023.csv: 62 rows and 8 columns
Loaded kroger_produce_raw.csv: 79 rows and 10 columns
Loaded vegetable-prices-2023.csv: 93 rows and 8 columns


## Inspecting data's statistics and values